# Imports

In [2]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt



from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler


from sklearn.linear_model import LogisticRegression

I (Keith) use sklearn: 1.4.2 imblearn: 0.12.3

In [3]:
import sklearn, imblearn
print("sklearn:", sklearn.__version__, "imblearn:", imblearn.__version__)

sklearn: 1.2.2 imblearn: 0.11.0


# Keep this random state

In [4]:
RANDOM_STATE=42

# Splitting functions and custom IQRClipper

In [5]:
# Convert one-hot cohort flags into a single label
def label_subcohort(df):
    df = df.copy()
    df["cohort"] = np.select(
        [
            df["is_mi"] == 1,
            df["is_pcr"] == 1,
            df["is_cbs"] == 1
        ],
        ["MI", "PCI", "CABG"],
        default="Other"
    )
    return df

# 1) collapse race to a few stable bins
def collapse_race(r):
    if pd.isna(r): return "UNKNOWN/OTHER"
    r = str(r).upper()
    if r.startswith("WHITE"): return "WHITE"
    if r.startswith("BLACK"): return "BLACK"
    if r.startswith("ASIAN"): return "ASIAN"
    if "HISPANIC" in r:        return "HISPANIC"
    if "PORTUGUESE" in r:      return "PORTUGUESE"
    if "SOUTH AMERICAN" in r:  return "HISPANIC"  # sensible merge
    if "PACIFIC ISLANDER" in r:return "PACIFIC"
    if r in {"OTHER","UNKNOWN","UNABLE TO OBTAIN","PATIENT DECLINED TO ANSWER",
             "MULTIPLE RACE/ETHNICITY","AMERICAN INDIAN/ALASKA NATIVE",
             "WHITE - OTHER EUROPEAN","WHITE - EASTERN EUROPEAN","WHITE - RUSSIAN",
             "WHITE - BRAZILIAN","BLACK/CAPE VERDEAN","BLACK/CARIBBEAN ISLAND"}:
        return "UNKNOWN/OTHER"
    return "UNKNOWN/OTHER"

def build_subject_strata(df: pd.DataFrame,
                         min_bin_size: int = 5,
                         outcome_col: str = "y",
                         cohort_col: str = "cohort",
                         race_col: str = "race"):
    
    g = df.copy()
    
    g["race_collapsed"] = g[race_col].map(collapse_race)

    # Make a *stable* per-subject cohort label.
    # Option A: majority cohort across their stays
    subj_mode_cohort = (
        g.groupby(["subject_id", cohort_col]).size()
         .reset_index(name="n")
         .sort_values(["subject_id","n"], ascending=[True, False])
         .drop_duplicates("subject_id")[["subject_id", cohort_col]]
         .rename(columns={cohort_col: "cohort_major"})
    )

    # Outcome as a subject-level flag (any positive stay)
    subj_y = (
        g.groupby("subject_id")[outcome_col]
         .max()
         .rename("y_any")
         .reset_index()
    )

    # Subject-level collapsed race: mode
    subj_race = (
        g.groupby(["subject_id","race_collapsed"]).size()
         .reset_index(name="n")
         .sort_values(["subject_id","n"], ascending=[True, False])
         .drop_duplicates("subject_id")[["subject_id","race_collapsed"]]
    )

    subj = (
        subj_mode_cohort.merge(subj_race, on="subject_id")
                        .merge(subj_y, on="subject_id")
    )

    # Compose strata label; you can add age bands, etc., if you want more control
    subj["strata"] = (
        subj["cohort_major"].astype(str) + "|" +
        subj["race_collapsed"].astype(str) + "|" +
        subj["y_any"].astype(str)
    )

    # Merge rare strata into a single bin to avoid stratify errors
    counts = subj["strata"].value_counts()
    rare = counts[counts < min_bin_size].index
    subj.loc[subj["strata"].isin(rare), "strata"] = "OTHER_BIN"

    return subj[["subject_id","strata"]]

def remove_outlier_drop_gapdays_and_split(df_all: pd.DataFrame,
                                          val_and_test_size: float = 0.30,
                                          random_state: int = 42,
                                          outcome_col: str = "y",
                                          cohort_col: str = "cohort",
                                          race_col: str = "race"):
    """
    - Hard-remove obvious data-entry outliers.
    - Drop prev_gap_days (non-imputable by design).
    - Stratified split by *subject_id* using a patient-level strata label
      (cohort_major × collapsed race × y_any), with rare bins merged.
    - Returns row-level train/val/test DataFrames.
    """
    
    
    df = df_all.copy()
    if "cohort" not in df.columns:
        df = label_subcohort(df)

    # 0) hard outliers + drop prev_gap_days
    df = df[(df["glucose_mean"] < 999999) & (df["wbc_last_24h"] < 400)]
    if "prev_gap_days" in df.columns:
        df = df.drop(columns=["prev_gap_days"])

    # 1) build subject-level strata
    subj = build_subject_strata(df, min_bin_size=5,
                                outcome_col=outcome_col,
                                cohort_col=cohort_col,
                                race_col=race_col)

    # 2) split BY subject IDs into train vs temp (val+test) with stratify
    train_ids, temp_ids = train_test_split(
        subj["subject_id"].values,
        test_size=val_and_test_size,
        random_state=random_state,
        stratify=subj["strata"].values
    )

    # 3) from the temp pool, split val vs test (50/50) with stratify again
    temp = subj[subj["subject_id"].isin(temp_ids)]
    val_ids, test_ids = train_test_split(
        temp["subject_id"].values,
        test_size=0.5,
        random_state=random_state,
        stratify=temp["strata"].values
    )

    # 4) map IDs back to rows
    train_df = df[df["subject_id"].isin(train_ids)].copy()
    val_df   = df[df["subject_id"].isin(val_ids)].copy()
    test_df  = df[df["subject_id"].isin(test_ids)].copy()

    # Optional sanity checks
    for s in ["train","val","test"]:
        print(s, df[df.subject_id.isin(eval(f"{s}_ids"))][outcome_col].mean())

    return train_df, val_df, test_df


def examine_splits(df, df_type='train'):
    print('-'*25, f'EXAMINING {df_type} data', '-'*25)
    print(f'{df_type} data has {df.shape[0]} ICU stays across {df.subject_id.nunique()} patients.')
    print('-'*25, 'EXAMINE FIRST FEW INSTANCES', '-'*25)
    display(df.head())
    
    print('-'*25, 'CLASS DISTRIBUTION (%)', '-'*25)
    display(df.y.value_counts(normalize=True)*100)
    
    print('-'*25, f'RACE DISTRIBUTION (%) ({df.race.nunique()} races)', '-'*25)
    display(df.race.value_counts(normalize=True)*100)
    
    print('-'*25, 'COHORT DISTRIBUTION (%)', '-'*25)
    display(df.cohort.value_counts(normalize=True)*100)
    
    print('-'*25, 'AGE DISTRIBUTION', '-'*25)
    df.hist('age')


class IQRClipper(BaseEstimator, TransformerMixin):
    """
    Clips feature values based on the Interquartile Range (IQR).
    Values below Q1 - factor*IQR are set to that lower bound,
    and values above Q3 + factor*IQR are set to that upper bound.

    Parameters
    ----------
    factor : float, default=1.5
        Multiplier for the IQR range.
    """

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X_df = pd.DataFrame(X)
        Q1 = X_df.quantile(0.25)
        Q3 = X_df.quantile(0.75)
        IQR = Q3 - Q1
        self.lower_bounds_ = Q1 - self.factor * IQR
        self.upper_bounds_ = Q3 + self.factor * IQR
        return self

    def transform(self, X):
        X_df = pd.DataFrame(X)
        X_clipped = X_df.clip(lower=self.lower_bounds_, upper=self.upper_bounds_, axis=1)
        # Return as same type as input
        return X_clipped if isinstance(X, pd.DataFrame) else X_clipped.to_numpy()

# REMOVE OBVIOUS OUTLIERS, DROP COLUMNS, THEN SPLIT

In [6]:
# REMOVE OBVIOUS OUTLIERS, DROP COLUMNS, THEN SPLIT

df = pd.read_csv("merged_chd_model_v2.csv")

train_df, val_df, test_df = remove_outlier_drop_gapdays_and_split(
    df,
    val_and_test_size = 0.3,
    random_state=RANDOM_STATE,
    outcome_col='y',
    cohort_col='cohort',
    race_col='race'
)

train 0.058043273753527753
val 0.06519823788546256
test 0.05415002219263205


Run this to see the distribution of train, val, and test splits

In [7]:
# examine_splits(train_df, df_type='train')
# examine_splits(val_df, df_type='val')
# examine_splits(test_df, df_type='test')

# 0. DROP IDENTIFIERS AND PREP THE DATA FOR HYPERPARAMETER TUNING

In [8]:
# 0. DROP IDENTIFIERS AND PREP THE DATA FOR HYPERPARAMETER TUNING

# Columns to drop from X
ID_COLS = ["subject_id", "hadm_id", "stay_id", "is_mi", "is_pcr", "is_cbs"]
DROP_COLS = [c for c in (ID_COLS + ["y"]) if c in train_df.columns]

# Build X_all / y_all and derive feature lists FROM X_all
X_all = pd.concat([train_df, val_df], axis=0).reset_index(drop=True).drop(columns=DROP_COLS)
y_all = pd.concat([train_df, val_df], axis=0).reset_index(drop=True)["y"]


# define numeric / categorical feature sets
numeric_cols = X_all.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = [c for c in X_all.columns if c not in numeric_cols]

# 1. DEFINE THE PIPELINE (PREPROCESSING & OVERSAMPLING & MODELLING)

To swap models, change the final pipeline step from LogisticRegression to your own estimator (e.g. MLP via scikeras). Keep preprocessing identical.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import RandomOverSampler
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, average_precision_score, classification_report, roc_auc_score, accuracy_score


num_pipe = Pipeline([
    ("iqr_clip", IQRClipper(factor=1.5)),
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ],
    remainder="drop"
)


X_train = train_df.drop(columns=ID_COLS + ["y"])
y_train = train_df["y"]

X_val = val_df.drop(columns=ID_COLS + ["y"])
y_val = val_df["y"]


model = XGBClassifier(
    objective="binary:logistic",
    eval_metric=["auc", "aucpr"],
    use_label_encoder=False,
    tree_method="hist",
    n_jobs=-1,
    random_state=RANDOM_STATE
)

pipe = ImbPipeline([
    ("pre", preprocessor),
    ("ros", RandomOverSampler(random_state=RANDOM_STATE)),
    ("model", model)
])


param_grid = {
    "model__n_estimators":    [500, 1000, 1500],
    "model__max_depth":       [6, 8],
    "model__learning_rate":   [0.01, 0.05],
    "model__subsample":       [0.8,0.9],
    "model__colsample_bytree":[0.8,0.9],
    "model__reg_lambda":      [1.0],
    "model__min_child_weight":[1],
}


aucpr_scorer = make_scorer(average_precision_score, needs_proba=True)


from sklearn.model_selection import PredefinedSplit


X_all = pd.concat([X_train, X_val])
y_all = pd.concat([y_train, y_val])
test_fold = [-1]*len(X_train) + [0]*len(X_val)  

cv = PredefinedSplit(test_fold=test_fold)

# ==== 7. Grid Search ====
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring=aucpr_scorer,
    cv=cv,
    verbose=2,
    n_jobs=-1,
    return_train_score=True
)

grid.fit(X_all, y_all)


print("Best Parameters:", grid.best_params_)
print("Best AUC-PR (val):", grid.best_score_)


DROP_COLS_TEST = [c for c in (ID_COLS + ["y"]) if c in test_df.columns]
Xte = test_df.drop(columns=DROP_COLS_TEST)
yte = test_df["y"].astype(int)

best_model = grid.best_estimator_
proba_te = best_model.predict_proba(Xte)[:, 1]
pred_te  = (proba_te >= 0.2).astype(int)

print("======  Test Set Evaluation ======")
print("Accuracy :", accuracy_score(yte, pred_te))
print("ROC AUC  :", roc_auc_score(yte, proba_te))
print("AUC-PR   :", average_precision_score(yte, proba_te))
print(classification_report(yte, pred_te, digits=3))


Fitting 1 folds for each of 48 candidates, totalling 48 fits


/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:28] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i

[CV] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=6, model__min_child_weight=1, model__n_estimators=500, model__reg_lambda=1.0, model__subsample=0.9; total time=   3.3s


/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:31] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.8, model__learning_rate=0.01, model__max_depth=6, model__min_child_weight=1, model__n_estimators=500, model__reg_lambda=1.0, model__subsample=0.8; total time=   3.1s


...

[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=6, model__min_child_weight=1, model__n_estimators=1000, model__reg_lambda=1.0, model__subsample=0.8; total time=   5.3s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=6, model__min_child_weight=1, model__n_estimators=1000, model__reg_lambda=1.0, model__subsample=0.9; total time=   5.4s


/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:17:54] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=8, model__min_child_weight=1, model__n_estimators=500, model__reg_lambda=1.0, model__subsample=0.8; total time=   4.0s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=8, model__min_child_weight=1, model__n_estimators=500, model__reg_lambda=1.0, model__subsample=0.9; total time=   4.1s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=6, model__min_child_weight=1, model__n_estimators=1500, model__reg_lambda=1.0, model__subsample=0.9; total time=   7.1s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=6, model__min_child_weight=1, model__n_estimators=1500, model__reg_lambda=1.0, model__subsample=0.8; total time=   7.1s
[CV] END model__colsample_bytree=0.9, model__learning_rate=0.05, model__max_depth=8, model__min_child_weight=1, model__n_estimators=1000, model__reg_lambda=1.0, model__subsample=0.8; total time=

/opt/anaconda3/lib/python3.11/site-packages/xgboost/training.py:199: UserWarning: [10:18:02] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Best Parameters: {'model__colsample_bytree': 0.9, 'model__learning_rate': 0.01, 'model__max_depth': 8, 'model__min_child_weight': 1, 'model__n_estimators': 500, 'model__reg_lambda': 1.0, 'model__subsample': 0.8}
Best AUC-PR (val): 0.16190354586574174
====== 📊 Test Set Evaluation ======
Accuracy : 0.6924101198402131
ROC AUC  : 0.7237731842973743
AUC-PR   : 0.1615108602164842
              precision    recall  f1-score   support

           0      0.971     0.696     0.811      2131
           1      0.106     0.631     0.182       122

    accuracy                          0.692      2253
   macro avg      0.538     0.664     0.496      2253
weighted avg      0.924     0.692     0.777      2253

